# Phase3: 拡張あり vs なし

**この回で学ぶこと**
- 基本的な拡張（切り抜き＋反転＋色調）で、過学習がどう変わるか
- **拡張は収束が遅くなる**ので、エポック数が少ないと効果が見えないこと

比較は「**拡張の有無だけ**」を変えます。モデル・学習率・エポック数・シードはそろえます（条件を1つだけ変える）。

> ⏱ 所要時間: 約3分（QUICK=True, 150エポック×2条件×1シード）。QUICK=Falseで約8分

In [ ]:
import sys
from pathlib import Path

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)

## 1. 学習中、画像はどう見えているか

`basic` 拡張では、**同じ1枚の画像がエポックごとに違う見え方**で学習に使われます。val/test は拡張しません。

In [ ]:
from torchvision import datasets

ds = datasets.ImageFolder(DATA / "train", T.make_transform("basic"))
shown = [ds[0][0] for _ in range(8)]        # 同じ1枚を8回取り出す＝8エポック分
show_images(shown, [f"エポック{i+1}" for i in range(8)], cols=4);

## 2. 拡張なし vs 拡張あり（basic）

In [ ]:
res = {}
for aug in ["none", "basic"]:
    print(f"== aug={aug} ==")
    res[aug] = run_seeds(SEEDS, aug=aug, epochs=EPOCHS)
df = pd.concat([d for d, _ in res.values()])
mean_table(df)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for row, aug in enumerate(["none", "basic"]):
    h = res[aug][1][0]["hist"]            # 最初のシードの履歴
    x = [e["epoch"] for e in h]
    axes[row, 0].plot(x, [e["train_loss"] for e in h], label="train")
    axes[row, 0].plot(x, [e["val_loss"] for e in h], label="val")
    axes[row, 0].set_title(f"{aug}: loss")
    axes[row, 1].plot(x, [e["train_acc"] for e in h], label="train")
    axes[row, 1].plot(x, [e["val_acc"] for e in h], label="val")
    axes[row, 1].set_title(f"{aug}: 精度")
    for a in axes[row]:
        a.legend(); a.grid(alpha=0.3)
fig.tight_layout();

## 3. 結果の読み方（MANUAL.mdの記載値・3シード平均）

| エポック | 条件 | train | val | test | gap |
|---|---|---|---|---|---|
| 60 | 拡張なし | 0.859 | 0.719 | 0.809 | 0.139 |
| 60 | 拡張あり | 0.756 | 0.715 | 0.757 | 0.041 |
| 150 | 拡張なし | 0.945 | 0.711 | 0.701 | 0.235 |
| 150 | 拡張あり | 0.822 | 0.732 | 0.796 | 0.089 |

- **60エポックでは拡張の効果が見えない**（gapは小さいが、精度は拡張なしと同程度）。拡張ありは**学習が遅い**。
- **150エポックで差が出る**。拡張なしは過学習して test が 0.701 まで悪化、拡張ありは 0.796 を保つ。
- → 拡張は「過学習を抑える代わりに収束が遅い」。**十分なエポックで比べないと効果を見誤る。**

## 🧪 試してみよう
1. `EPOCHS = ep(60)` に変えて再実行しよう。拡張の効果は見える？（上の表の60エポックと比べる）
2. `QUICK = False` にして3シードで比べよう。**差は標準偏差より大きい？**（val/test は約77枚で誤差が大きい）
3. `T.AUG_SETS["basic"]` の `scale=(0.6, 1.0)` を `(0.3, 1.0)` に変えると、学習曲線はどうなる？
4. 拡張なしのほうが test が低いのに、**train 精度は拡張なしのほうが高い**。これは何を意味する？

## まとめ
- 拡張で train-val の差（過学習）は縮み、十分に学習すれば test も改善する。
- ただし**収束が遅い**ので、短いエポックで比べない。
- 比較は**複数シード平均**で。

➡ 次は `04_ablation.ipynb`（どの拡張が効くのか切り分ける）